# Discrete Poisson equation: complete native workflow

Keep the original 8 by 8 grid, 64 normalized source amplitudes, six QPE bits,
reciprocal table, postselection, source normalization and both contour figures.
All quantum construction uses public FlagQuantum Python functions.

The independent Circuit checks complete complex amplitudes, including absolute
phase. Matching the source quantum program is distinct from solving the stated
finite-difference problem accurately: the source normalization and axis convention
produce a substantial discrepancy, which is measured rather than corrected away.
No accuracy, optimality or speed advantage is claimed.


In [ ]:
import math
from pathlib import Path
from typing import Annotated

from examples.synthesis.algorithms.quantum_differential_equations_solvers.discrete_poisson_solver.overview.physics import (
    AMPLITUDES,
    ANGLES,
    COSINE,
    EIGENVALUES_X,
    EIGENVALUES_Y,
    GRID,
    MAX_EIG,
    OPERATOR,
    PREFACTOR,
    QPE_SIZE,
    SINE,
    SOURCE,
    ZGRID,
    finite_difference_problem,
    walsh_coefficients,
)
from examples.synthesis.algorithms.quantum_differential_equations_solvers.discrete_poisson_solver.overview.workflow import (
    extract_solution,
    plot_solution,
    plot_source,
    verify_program,
)
from IPython.display import Image, display

import flagquantum as fq
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QArray,
    QBit,
    QNum,
    allocate,
    apply,
    inplace_prepare_amplitudes,
    qfunc,
    qpe,
    select_rotation,
    suzuki_trotter,
    unitary,
    within,
)

output_directory = Path("artifacts/development/discrete_poisson")


In [ ]:
coefficients_x = walsh_coefficients(EIGENVALUES_X)
coefficients_y = walsh_coefficients(EIGENVALUES_Y)
print("Original ordered Walsh expansions:", coefficients_x, coefficients_y)
assert len(coefficients_x) == len(coefficients_y) == 8
print("Fresh Hamiltonian:", OPERATOR)


In [ ]:
@qfunc
def powered_evolution(power: int, target: QArray[QBit]) -> None:
    suzuki_trotter(OPERATOR, -2 * math.pi * power / MAX_EIG, 1, 1, target)


In [ ]:
@qfunc
def spatial_transform(target: QArray[QBit]) -> None:
    unitary(SINE, target[:3])
    unitary(COSINE, target[3:])


The current native transform functions require complete registers. Explicit
normalized spatial matrices on static views express the same fixed transforms.
The native QPE estimator is a six-bit fractional number; its zero postselection
and physical words agree with the source integer register. The reciprocal table
uses source **integer words**: entry k is 1/(64 k), not 1/k. This workflow does
not claim generic QCallable, arrays of numeric registers, fresh Output helpers,
or integer-typed QPE output support.


The source inverse helper combines QPE and rotation. Native QPE currently
owns hidden storage at the program root, so the final root below keeps QPE and
its inverse region visible. Static evolution and spatial helpers remain composed.
This is an explicit construction boundary, not a generic helper support claim.


In [ ]:
@qfunc
def reciprocal_rotation(estimator: Annotated[QNum, (6, False, 6)], flag: QBit) -> None:
    select_rotation("Y", ANGLES, estimator, flag)


In [ ]:
print("Grid:", GRID)
print("Source samples:", ZGRID.reshape(8, 8))
print("Normalized RHS:", SOURCE.reshape(8, 8))
assert len(AMPLITUDES) == 64
assert abs(float(SOURCE.norm()) - 1) < 1e-14


In [ ]:
source_figure = plot_source(output_directory)
display(Image(filename=str(source_figure)))


In [ ]:
print("Fresh spectral Hamiltonian:", OPERATOR)


In [ ]:
assert QPE_SIZE == 6 and PREFACTOR == 1 / 64
assert math.isclose(MAX_EIG, math.pi**2)
print("Original scale:", PREFACTOR / MAX_EIG)


In [ ]:
@qfunc
def poisson(
    amplitudes: list[float],
    xy: Output[QArray[QBit]],
    phase: Output[Annotated[QNum, (6, False, 6)]],
    indicator: Output[QBit],
) -> None:
    allocate(6, xy)
    allocate(phase)
    allocate(indicator)
    inplace_prepare_amplitudes(amplitudes, 0, xy)
    with within():
        spatial_transform(xy)
    with apply():
        with within():
            qpe(lambda power: powered_evolution(power, xy), phase)
        with apply():
            reciprocal_rotation(phase, indicator)


program = synth.synthesize(
    poisson,
    classical_inputs={"amplitudes": list(AMPLITUDES)},
    max_qubits=13,
    max_workspace=0,
    memory_limit_bytes=64 * 1024**2,
)
check = verify_program(program)
print("Complete independent Circuit validation:", check)
print("Actual static resources:", program.report["resources"])


In [ ]:
result = fq.run(program)
print(
    "Named numeric probabilities:",
    dict(program.probabilities(result, registers=("indicator",))),
)


In [ ]:
solution = extract_solution(program)
print("Single global phase:", solution.global_phase)
print("Success probability:", solution.success)
print("Retained finite-QPE leakage:", solution.retained_phase_leakage)


In [ ]:
result_matrix = solution.quantum
print("Source-normalized result:", result_matrix)


In [ ]:
assert float(result_matrix[4, 4]) >= 0
print("Center-positive source sign convention retained")


In [ ]:
classical_operator, classical_rhs, classical_result_matrix = finite_difference_problem()
print("Classical solution:", classical_result_matrix)
print("Relative distance:", solution.relative_distance)
print("Transpose-only diagnostic:", solution.transpose_relative_distance)
print("Relative residual:", solution.relative_residual)
assert (
    float(
        (classical_operator @ classical_result_matrix.flatten() - classical_rhs).norm()
    )
    < 1e-13
)


In [ ]:
solution_figure = plot_solution(solution, output_directory)
display(Image(filename=str(solution_figure)))


The source-scale discrepancy remains visible; no fitted scalar, transpose,
resized grid or increased precision is substituted to make the result appear
accurate. Retained estimator leakage is algorithmic output, not dirty compiler
workspace. The quantum oracle acceptance tests compare complete physical states;
the classical diagnostics separately expose limitations of the source example.
